In [5]:
import sys
import pandas as pd
import numpy as np
import IPython
from IPython.display import display

print( f"Python {sys.version}" )
print( f"Pandas {pd.__version__}" )
print( f"NumPy {np.__version__}" )
print( f"IPython {IPython.__version__}" )

Python 3.13.14 (main, Jul 14 2026, 04:45:36) [GCC 14.2.0]
Pandas 3.0.5
NumPy 2.5.1
IPython 9.16.1


In [6]:
# Load the pokemon dataset
df_pokemon = pd.read_csv('https://raw.githubusercontent.com/ShaileshDhama/Exploratory-Data-Analysis-On-Pokemon-Dataset/master/Complete%20Pokemon.csv')
df_pokemon.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   abilities          801 non-null    str    
 1   against_bug        801 non-null    float64
 2   against_dark       801 non-null    float64
 3   against_dragon     801 non-null    float64
 4   against_electric   801 non-null    float64
 5   against_fairy      801 non-null    float64
 6   against_fight      801 non-null    float64
 7   against_fire       801 non-null    float64
 8   against_flying     801 non-null    float64
 9   against_ghost      801 non-null    float64
 10  against_grass      801 non-null    float64
 11  against_ground     801 non-null    float64
 12  against_ice        801 non-null    float64
 13  against_normal     801 non-null    float64
 14  against_poison     801 non-null    float64
 15  against_psychic    801 non-null    float64
 16  against_rock       801 non-null    fl

# 2. Pandas: Transform data

This section will focus on `transform()` and `apply()`.

In [8]:
# Prepare the data to play around
df = df_pokemon[ ['pokedex_number','name','type1','type2','hp','attack','defense'] ]
df

,pokedex_number,name,type1,type2,hp,attack,defense
0,1,Bulbasaur,grass,poison,45,49,49
1,2,Ivysaur,grass,poison,60,62,63
2,3,Venusaur,grass,poison,80,100,123
3,4,Charmander,fire,NaN,39,52,43
4,5,Charmeleon,fire,NaN,58,64,58
...,...,...,...,...,...,...,...
796,797,Celesteela,steel,flying,97,101,103
797,798,Kartana,grass,steel,59,181,131
798,799,Guzzlord,dark,dragon,223,101,53
799,800,Necrozma,psychic,NaN,97,107,101


In [ ]:
# When a transform function IS NOT an aggregate function
plus20 = lambda x: x+20

# Both commands produce the same result (pd.Series), use only one
# ถ้า function ที่ pass เข้าไป ไม่มี risk ในการยุบข้อมูล เพราะ apply() จะ return pd.Series ที่มีจำนวน row เท่ากับ input
df['hp'].apply(plus20) 
df['hp'].transform(plus20) 

# ทั้ง 2  function อนญาตให้เราสามารถ pass function เข้าไปได้ทั้งแบบเป็นชื่อ function และเป็น lambda function

0       65
1       80
2      100
3       59
4       78
      ... 
796    117
797     79
798    243
799    117
800    100
Name: hp, Length: 801, dtype: int64

In [ ]:
# When a transform function IS an aggregate function

# transform() cannot produce aggregated results BUT apply() can.
print( "apply('mean') :" , df['hp'].apply('mean') )        # OK

# ถ้า function ที่ pass เข้าไปมี risk ในการยุบข้อมูล เช่น aggregate function เช่น mean() ถ้าใช้ transform จะขึ้น error มา
print( "transform('mean') :", df['hp'].transform('mean') ) # ValueError

apply('mean') : 68.95880149812734


ValueError: Function did not transform

For pandas.DataFrame:

In [ ]:
# When a transform function IS NOT an aggregate function

# Copy the original values just for easy comparison
df2 = df.loc[ :, ['hp','attack','defense'] ]

# The two commands below produce the same outputs (pandas.DataFrame)
# Be careful that both commands don't alter the original dataframe (df)
df2[['hp_after','attack_after','defense_after']] = df[['hp','attack','defense']].apply(lambda x: x+20)

#df2[['hp_after','attack_after','defense_after']] = df[['hp','attack','defense']].transform(lambda x: x+20)

df2

,hp,attack,defense,hp_after,attack_after,defense_after
0,45,49,49,65,69,69
1,60,62,63,80,82,83
2,80,100,123,100,120,143
3,39,52,43,59,72,63
4,58,64,58,78,84,78
...,...,...,...,...,...,...
796,97,101,103,117,121,123
797,59,181,131,79,201,151
798,223,101,53,243,121,73
799,97,107,101,117,127,121


In [12]:
# When a transform function IS an aggregate function
# transform() cannot produce aggregated results BUT apply() can
# The output of transform() has to be pandas.DataFrame that has the same shape as self

display( df[['hp','attack','defense']].apply('mean') )      # OK
display( df[['hp','attack','defense']].transform('mean') )  # ValueError

hp         68.958801
attack     77.857678
defense    73.008739
dtype: float64

ValueError: Function did not transform

`apply()` vs. `transform()`
>สองอันนี้จะมีความสำคัญที่แตกต่างกันในอนาคต เช่น การนำมาใช้ groupby()
- Comparison 1:
  - `apply()` supports both aggregation and non-aggregation functions.
  - `transform()` doesn't support aggregation functions (causing an error).
- Comparison 2:
  - <u>`apply()` sends the entire pandas.DataFrame to the calling function</u> so it can work with multiple Series at a time.
  - <u>`transform()` only sends each individual column as a pandas.Series to the calling function</u> so is can only work with one Series at a time. Also, the result from `transform()` has the same shape as self.


In [ ]:
# apply() allows working with multiple pandas.Series at a time
# apply() สามารถทำงานกับ data ที่ pass เข้ามาในรูปแบบ dataframe ได้ ไม่มีปัญหา

df = df_pokemon.loc[ :, ['pokedex_number','name','type1','type2','hp','attack','defense'] ]
df['att_sub_def'] = df[['hp','attack','defense']].apply(lambda x: x['attack'] - x['defense'], axis=1) # OK
df

,pokedex_number,name,type1,type2,hp,attack,defense,att_sub_def
0,1,Bulbasaur,grass,poison,45,49,49,0
1,2,Ivysaur,grass,poison,60,62,63,-1
2,3,Venusaur,grass,poison,80,100,123,-23
3,4,Charmander,fire,NaN,39,52,43,9
4,5,Charmeleon,fire,NaN,58,64,58,6
...,...,...,...,...,...,...,...,...
796,797,Celesteela,steel,flying,97,101,103,-2
797,798,Kartana,grass,steel,59,181,131,50
798,799,Guzzlord,dark,dragon,223,101,53,48
799,800,Necrozma,psychic,NaN,97,107,101,6


In [ ]:
# transform() can only work with one pandas.Series at a time
# function transform จะถึงข้อมูลออกมาทีละ series (row) เพราะฉะนั้นการเขียน function ในบริบท data frame จะ error
# transform จะ extract ข้อมูลออกมา แล้วค่อย pass เข้าไปใน function
# ถ้ามี interact เช่น +,- ระหว่าง column จะมีปัญหาได้ error

df = df_pokemon.loc[ :, ['pokedex_number','name','type1','type2','hp','attack','defense'] ]
df['att_sub_def'] = df[['hp','attack','defense']].transform(lambda x: x['attack'] - x['defense'], axis=1) # ValueError
df

ValueError: Function did not transform